
Known Datasets

- https://api.cerebromcp.com/characters
- https://api.cerebromcp.com/tactics
- https://api.cerebromcp.com/crisis
- https://api.cerebromcp.com/gems
- https://api.cerebromcp.com/mctcodes

# Infinity Gems

The source for Infinity Gems is straight forward and can be handled in a single cell. This should be done before loading the character data so the MCT codes for Gems can be referenced.

In [0]:
"""Load Infinity Gems

Parameters
----------
col_list : list
    list of columns to be loaded into the DataFrame
col_rename : list
    list of column names to use that match destination table
df_gems : DataFrame
    DataFrame to hold and transform contents of the input file
gem_info_json : list
    json containing infinity gems
gem_path : str
    url to the api to get the gem data
schema : StructType
    schema of the destination table
table_name : str
    destionation table for the imported data
"""

import requests
import json

from pyspark.sql import functions as F

# input variables
gem_path = 'https://api.cerebromcp.com/gems'
table_name = "mcp.game.infinity_gems"
schema = spark.table(table_name).schema  # StructType

# fields to import from json file
col_list = [
  'CID',
  'Name',
  'ID',
  'Color',
  'Cost',
]

# name of fields in destination table
col_rename = [
  'gem_id',
  'gem_name',
  'alternate_gem_id',
  'color',
  'threat',
]

# get json for infinity gems
gem_info_json = requests.get(gem_path).json()

# create dataframe and rename the fields
df_gems = spark.createDataFrame(gem_info_json).select(*col_list).toDF(*col_rename)

# cast every column to matching datatype and set column order to match table
df_gems = df_gems.select([
    F.col(f.name).cast(f.dataType).alias(f.name) 
    for f in schema.fields
    ])

# write to table
df_gems.write.mode("overwrite").saveAsTable(table_name)

# show data
display(df_gems)

# Characters

In [0]:
"""Load Character Data

Parameters
----------
char_info_json : list
    json containing characters
char_path : str
    url to the api to get the gem data
df : DataFrame
    DataFrame to hold and transform contents of the input file
"""

import requests
import json

from pyspark.sql.functions import when, col

# input variables
char_path = 'https://api.cerebromcp.com/characters'

# get json for characters
char_info_json = requests.get(char_path).json()

# create dataframe with charcters
df = spark.createDataFrame(char_info_json)

# replace blanks with NULL
df = df.select([
  when(col(c) == "", None).otherwise(col(c)).alias(c)
  if dict(df.dtypes)[c] == 'string'
  else col(c) for c in df.columns
  ])

display(df)

In [0]:
"""Create Temporary Character IDs
    Need temporary MCT codes for cards that are missing a CID.

Parameters
----------
missing_count : int
    gets count of records with missing CID
df_missing_cid : DataFrame
    DataFrame to hold missing CIDs
df_with_new_cid: DataFrame
    DataFrame to hold cards with new CIDs
w : WindowSpec

"""
# get the count of rows missing a CID
missing_count = df.filter((col("cid").isNull()) | (col("cid") == "")).count()

# if any rows are missing a CID then create a temporary CID
if missing_count > 0:
    from pyspark.sql.window import Window
    from pyspark.sql.functions import (
        col, when, row_number, lpad, concat, lit, monotonically_increasing_id
    )

    # Identify rows with missing or blank CID
    w = Window.orderBy(monotonically_increasing_id())
    df_missing_cid = (
        df.filter((col("cid").isNull()) | (col("cid") == ""))
        .withColumn("new_cid_seq", row_number().over(w))
        .withColumn("new_cid", concat(lit("T"), lpad(col("new_cid_seq"), 7, "0")))
        .drop("new_cid_seq")
    )

    # Update df with new CID where missing
    df_with_new_cid = (
        df.join(
            df_missing_cid.select("name", "new_cid"),
            on="name",
            how="left"
        )
        .withColumn(
            "cid",
            when((col("cid").isNull()) | (col("cid") == ""), col("new_cid")).otherwise(col("cid"))
        )
        .drop("new_cid")
    )

    df = df_with_new_cid
else:
    print('no missing CIDs')

In [0]:
"""Load Affiliations
    There is not a separate api for affiliations,
    so we have to parse the Affiliations column.

Parameters
----------
df_affiliations : DataFrame
    DataFrame to hold a unique list of affiliations
schema : StructType
    schema of the destination table
table_name : str
    destination table for the imported data
"""

from pyspark.sql.functions import split, explode, trim, col
from pyspark.sql import functions as F

table_name = "mcp.game.affiliations"
schema = spark.table(table_name).schema  # StructType

df_affiliations = (
    df.select(explode(split(col("Affiliations"), ",")).alias("affiliation_name"))
    .select(trim(col("affiliation_name")).alias("affiliation_name"))
    .filter(
        (col("affiliation_name").isNotNull()) &
        (col("affiliation_name") != "")
    )
    .distinct()
    .orderBy("affiliation_name")
)

# Cast every column to matching datatype and set column order to match table
df_affiliations = df_affiliations.select(
    [F.col(f.name).cast(f.dataType).alias(f.name) for f in schema.fields]
)

df_affiliations.write.mode("overwrite").saveAsTable(table_name)

display(df_affiliations)

In [0]:
display(df)

In [0]:
"""Extract all CP and MCP codes from the CP column

Parameters
----------
df_skus : DataFrame
    DataFrame containing all unique CP/MCP codes found in the data
"""

from pyspark.sql.functions import regexp_extract_all, explode, upper, col, lit

# Extract all CP and MCP codes (case insensitive pattern)
# Pattern matches 'CP' or 'MCP' followed by one or more digits
df_skus = (
    df.select(
        explode(
            regexp_extract_all(upper(col("CP")), lit(r"(M?CP\d+)"))
        ).alias("sku")
    )
    .filter(col("sku") != "")  # Remove empty strings
    .distinct()
    .orderBy("sku")
)

display(df_skus)

In [0]:
"""Map product skus codes to character IDs

Parameters
----------
df_sku_to_character_id : DataFrame
    DataFrame mapping each CP/MCP code to the character IDs that reference it
"""

from pyspark.sql.functions import regexp_extract_all, explode, upper, col, lit

# Extract all CP/MCP codes along with their character ID
df_sku_to_character_id = (
    df.select(
        col("cid").alias("character_id"),
        explode(
            regexp_extract_all(upper(col("CP")), lit(r"(M?CP\d+)"))
        ).alias("sku")
    )
    .filter(col("sku") != "")  # Remove empty strings
    .orderBy("sku", "character_id")
)

display(df_sku_to_character_id)

In [0]:
"""Load Characters

Parameters
----------
col_list : list
    list of columns to be loaded into the DataFrame
col_rename : list
    list of column names to use that match destination table
df_char : DataFrame
    DataFrame to hold a unique list of affiliations
schema : StructType
    schema of the destination table
table_name : str
    destionation table for the imported data
"""

from pyspark.sql.functions import col, lit
from pyspark.sql import functions as F

table_name = "mcp.game.characters"
schema = spark.table(table_name).schema  # StructType

col_list = [
  'cid',
  'Name',
  'Alias',
  'Cost',
  'front_health',
  'back_health',
  'extraPower',
  'Leadership',
  'Leadership_Cards',
  'GemLimit'
]

col_rename = [
  'character_id',
  'character_name',
  'alter_ego',
  'threat',
  'front_health',
  'back_health',
  'extra_power',
  'leadership_affiliation_id',
  'leadership_card_id',
  'gem_limit',
]

df_char = df.select(*col_list).toDF(*col_rename)
df_char = df_char.withColumn("gem_bearer", col("gem_limit") > 0)
df_char = df_char.withColumn("effective_date", lit(None).cast("date"))
df_char = df_char.withColumn("update_key", lit(None).cast("string"))
df_char = df_char.withColumn("current_version", lit(True))

# cast every column to matching datatype and set column order to match table
df_char = df_char.select([
    F.col(f.name).cast(f.dataType).alias(f.name) 
    for f in schema.fields
    ])

df_char.write.mode("overwrite").saveAsTable(table_name)

display(df_char)

In [0]:
"""Load Character Affiliation
Parse the character data to get one row per character and affiliation

Parameters
----------
df_char_affiliations : DataFrame
    DataFrame to hold a unique list of affiliations
schema : StructType
    schema of the destination table
table_name : str
    destination table for the imported data
"""

from pyspark.sql.functions import split, explode, trim, col
from pyspark.sql import functions as F

table_name = "mcp.game.affiliated_charcters"
schema = spark.table(table_name).schema  # StructType

df_char_affiliations = (
    df.select(
        split(col("Affiliations"), ",").alias("affiliation_name"),
        col("cid").alias("character_id")
    )
    .withColumn("affiliation_name", explode(col("affiliation_name")))
    .withColumn("affiliation_name", trim(col("affiliation_name")))
    .filter(
        (col("affiliation_name").isNotNull()) &
        (col("affiliation_name") != "")
    )
    .distinct()
    .orderBy("affiliation_name", "character_id")
)

# Cast every column to matching datatype and set column order to match table
df_char_affiliations = df_char_affiliations.select([
    F.col(f.name).cast(f.dataType).alias(f.name)
    for f in schema.fields
])

# Write to table
df_char_affiliations.write.mode("overwrite").saveAsTable(table_name)

# Show data
display(df_char_affiliations)

In [0]:
"""Load Character Gems
Parse the character data to get one row per character and Gem

Parameters
----------
df_char_gems : DataFrame
    DataFrame to hold a unique list of affiliations
schema : StructType
    schema of the destination table
table_name : str
    destination table for the imported data
"""

from pyspark.sql.functions import split, explode, trim, col, regexp_replace
from pyspark.sql import functions as F

table_name = "mcp.game.charcters_gems"
schema = spark.table(table_name).schema  # StructType

df_char_gems = (
    df.withColumn("Gems_clean", regexp_replace(col("Gems"), r"[\[\]]", ""))
    .select(
        split(col("Gems_clean"), ",").alias("alternate_gem_id"),
        col("cid").alias("character_id")
    )
    .withColumn("alternate_gem_id", explode("alternate_gem_id"))
    .withColumn("alternate_gem_id", trim(col("alternate_gem_id")))
    .filter(
        (col("alternate_gem_id").isNotNull()) &
        (col("alternate_gem_id") != "")
    )
    .distinct()
    .orderBy("alternate_gem_id", "character_id")
)

# Join to df_gems to get the mct code for the gems
df_char_gems = df_char_gems.join(
    df_gems.select("alternate_gem_id", "gem_id"),
    on="alternate_gem_id",
    how="left"
).drop("alternate_gem_id")

# Cast every column to matching datatype and set column order to match table
df_char_gems = df_char_gems.select([
    F.col(f.name).cast(f.dataType).alias(f.name)
    for f in schema.fields
])

# Write to table
df_char_gems.write.mode("overwrite").saveAsTable(table_name)

display(df_char_gems)

In [0]:
%sql
INSERT OVERWRITE mcp.game.product_codes
SELECT
  x.sku
  ,CAST(
      CASE x.release_date
      WHEN '1900-1-0' THEN NULL
      ELSE x.release_date END 
      as date) release_date
FROM
  read_files('/Volumes/mcp/default/uploads/sku_dates.csv') x

In [0]:
table_name = "mcp.game.product_codes"
schema = spark.table(table_name).schema  # StructType

# Get existing SKUs from the table
df_existing_skus = spark.table(table_name).select("sku")

# Find new SKUs that don't exist in the table
df_new_skus = df_skus.join(df_existing_skus, on="sku", how="left_anti")

# Only append if there are new SKUs
if df_new_skus.count() > 0:
    print(f"Adding {df_new_skus.count()} new SKUs")
    df_new_skus.write.mode("append").saveAsTable(table_name)
else:
    print("No new SKUs to add")

In [0]:
table_name = "mcp.game.product_to_character"
schema = spark.table(table_name).schema  # StructType

df_sku_to_character_id.write.mode("overwrite").saveAsTable(table_name)

In [0]:
from pyspark.sql import functions as F

table_name = "mcp.game.character_updates"
schema = spark.table(table_name).schema  # StructType

update_202511 = [
    '00240101',
    '00530101',
    '00330101',
    '01030101',
    '01030102',
    '01510101',
    '00550101',
    '00240102',
    '00510101',
    '01260102',
    '00510102',
    '01600101',
    '00180101',
]
update_202505 = [
    '01000101',
    '00280101',
    '01120101',
    '00300101',
    '00230101',
    '00570101',
    '00670101',
    '00500101',
    '00670102',
    '00600101',
    '01020101',
    '00570102',
    '00710102',
    '00420102',
    '01120102',
]
update_202502 = [
    '00250107',
    '00910104',
]
update_202309 = [
    '00370101',
    '00340101',
    '00370102',
    '00480101',
    '00300102',
    '00410101',
    '00570101',
    '00470101',
    '00450102',
    '00650101',
    '00790101',
    '00120101',
    '00410102',
    '00200102',
    '00930101',
    '00530102',
    '00320102',
    '00250107',
    '00110101',
    '00130102',
    '00230102',
    '00400102',
]
update_202111 = [
    '00070101',
    '00300101',
    '00010103',
    '00010104',
    '00160101',
    '00280102',
    '00190102',
    '00150101',
    '00270101',
    '00210101',
    '00170101',
    '00040101',
    '00340102',
    '00050101',
    '00290101',
    '00010107',
    '00080101',
    '00150102',
    '00170102',
    '00080102',
    '00130102',
    '00110102',
    '00010110',
    '00180101',
]

df_update_202511 = spark.createDataFrame(
    [(str(202511), cid) for cid in update_202511],
    ["update_key", "character_id"]
)

df_update_202505 = spark.createDataFrame(
    [(str(202505), cid) for cid in update_202505],
    ["update_key", "character_id"]
)

df_update_202502 = spark.createDataFrame(
    [(str(202502), cid) for cid in update_202502],
    ["update_key", "character_id"]
)

df_update_202309 = spark.createDataFrame(
    [(str(202309), cid) for cid in update_202309],
    ["update_key", "character_id"]
)

df_update_202111 = spark.createDataFrame(
    [(str(202111), cid) for cid in update_202111],
    ["update_key", "character_id"]
)

df_update_all = (
    df_update_202511
    .unionByName(df_update_202505)
    .unionByName(df_update_202502)
    .unionByName(df_update_202309)
    .unionByName(df_update_202111)
)

# cast every column to matching datatype and set column order to match table
df_update_all = df_update_all.select([
    F.col(f.name).cast(f.dataType).alias(f.name) 
    for f in schema.fields
    ])

# write to table
df_update_all.write.mode("overwrite").saveAsTable(table_name)

display(df_update_all)

In [0]:
%sql
-- Batch updates for performance

UPDATE mcp.game.characters
SET
  front_health = CASE character_id
    WHEN '01520101' THEN 5
    WHEN '01070101' THEN 7
    WHEN '01520102' THEN 5
    WHEN '00230101' THEN 6
    WHEN '00930101' THEN 9
    WHEN '01090101' THEN 5
    WHEN '01660104' THEN 6
    WHEN '30090103' THEN 7
    WHEN '00200102' THEN 7
    WHEN '00510102' THEN 8
    WHEN '00010109' THEN 6
    WHEN '01620102' THEN 6
    WHEN '00110101' THEN 7
    WHEN '01090102' THEN 5
    ELSE front_health
  END,
  back_health = CASE character_id
    WHEN '01520101' THEN 5
    WHEN '01070101' THEN 8
    WHEN '01520102' THEN 6
    WHEN '00230101' THEN 7
    WHEN '00930101' THEN 8
    WHEN '01090101' THEN 5
    WHEN '01660104' THEN 6
    WHEN '30090103' THEN 6
    WHEN '00200102' THEN 7
    WHEN '00510102' THEN 6
    WHEN '00010109' THEN 5
    WHEN '01620102' THEN 6
    WHEN '00110101' THEN 8
    WHEN '01090102' THEN 5
    ELSE back_health
  END
WHERE character_id IN (
  '01520101','01070101','01520102','00230101','00930101','01090101','01660104','30090103',
  '00200102','00510102','00010109','01620102','00110101','01090102'
);

-- Set Back of Hulks to NULL so 0 doesn't impact statistical calcs
UPDATE mcp.game.characters
SET back_health = NULL
WHERE character_name ilike '%hulk%'
  AND character_name not ilike '%hulkbuster%';

In [0]:
%sql
--Sets the effective date to the last update date or release date if no updates
MERGE INTO mcp.game.characters AS target
USING 
  (
    SELECT
      c.character_id
      ,CASE WHEN MAX(au.effective_date) IS NOT NULL 
        THEN MAX(au.effective_date)
        ELSE MIN(p.release_date)
        END use_date
      ,max(au.update_key) update_key
      --,row_number() OVER(PARTITION BY c.character_id ORDER BY p.release_date) AS row_no
    FROM
      mcp.game.characters c
      INNER JOIN mcp.game.product_to_character pc
        ON c.character_id = pc.character_id
      INNER JOIN mcp.game.product_codes p
        ON pc.sku = p.sku
      LEFT JOIN mcp.game.character_updates cu
        ON c.character_id = cu.character_id
      LEFT JOIN mcp.game.amg_updates au
        ON cu.update_key = au.update_key
    GROUP BY
       c.character_id
  ) AS source
ON 
  target.character_id = source.character_id
WHEN MATCHED THEN
  UPDATE SET 
     target.effective_date = source.use_date
    ,target.update_key = source.update_key